# Lab: Mid Semeter Capstone Lab
## Disease Outbreak Response
---
## Case Study: Containing a Contagious Disease in a Metropolitan Area
### Scenario
You are developing an AI system to help authorities contain a sudden outbreak in a large metropolitan region.

The system integrates multiple AI techniques:

- Grid-based search to plan travel between hospitals and outbreak zones (BFS/DFS)
- Heuristic-based graph search for fastest patient transfers (GBFS, A*)
- Minimax and alpha-beta pruning to allocate resources optimally under uncertain outbreak dynamics
- Propositional and first-order logic to model rules, patients, and hospitals
- Probabilistic reasoning to estimate risk and success of treatments

The metropolitan area is represented via complex datasets: grids, graphs, resources, and probability tables. Each section of this lab focuses on one AI technique, but the outputs will integrate for a full outbreak response plan.

### Input File Overview

This lab uses multiple files, each representing a real-world dataset:

1. **regions.txt**  
   - Grid of the metropolitan area with hospitals (S), outbreak zones (G), blocked regions (X), and open zones (.)  
   - Students must implement BFS/DFS to find paths.

2. **graph.txt**  
   - Weighted graph representing road network between hospitals, patients, and outbreak centers  
   - Format: `from_city cost to_city`  
   - Used for GBFS and A* searches

3. **heuristics.txt**  
   - Heuristic estimates of distance or risk to outbreak for each node  
   - Format: `node_id heuristic_value`  
   - Used in GBFS and A* searches

4. **game_state.json**  
   - Represents MAX (disease control team) and MIN (disease progression)  
   - Contains resources, patient counts, and current turn  
   - Used in Minimax/Alpha-Beta section

5. **sensors.txt**
   - Sensors indicate outbreak symptoms in regions  

6. **rules.txt**  
   - Rules combine sensor values to generate alerts

7. **kb.txt**  
   - Knowledge base of patients, diseases, and treatment rules  
   - Used in propositional logic / resolution / inference

8. **patients.txt / hospitals.txt**  
   - FOL entities representing patient severity and hospital capacity  
   - Used in first-order logic queries

9. **probabilities.csv**  
   - Conditional probabilities of treatment outcomes  
   - Used for uncertainty modeling

## Learning Objectives

By completing this lab, students will:

- Implement BFS, DFS, GBFS, and A* search on complex datasets
- Understand priority queues and heuristic-driven exploration
- Apply Minimax and Alpha-Beta pruning in a resource allocation scenario
- Model real-world knowledge using propositional and first-order logic
- Apply probabilistic reasoning to estimate risks and treatment outcomes
- Integrate multiple AI techniques into a single response system

## Program Requirements

You must implement the following functions/classes:

- `read_grid(filename)` → loads `regions.txt`
- `bfs(grid, start, goal)` → returns path and nodes explored
- `dfs(grid, start, goal)` → returns path and nodes explored
- `read_graph(filename)` → loads `graph.txt` into adjacency list
- `read_heuristic(filename)` → loads heuristic values
- `greedy_best_first_search(graph, heuristic, start, goal)`
- `a_star(graph, heuristic, start, goal)`
- `GameState` class → model current resources, patients, turn
- `minimax(state, depth, maximizingPlayer)` → returns best action
- `alpha_beta(state, depth, alpha, beta, maximizingPlayer)` → returns best action
- `Sensor` / `Rule` classes → evaluate outbreak alerts
- `KnowledgeBase` class → CNF conversion, resolution, query
- `FOL_KB` class → simulate first-order logic queries
- `ProbabilityEngine` class → compute marginals, conditionals, Bayes probabilities
- `write_output(path, results, filename)` → unified output

Each function/class should include comments explaining logic and pseudocode translation.

In [18]:
# Example for Graphs
import heapq

class Graph:
    def __init__(self):
        self.nodes = {}
        self.adjacency = {}

    def add_node(self, node_id, **attrs):
        self.nodes[node_id] = attrs
        self.adjacency[node_id] = []

    def add_edge(self, node1, node2, cost):
        self.adjacency[node1].append((node2, cost))
        self.adjacency[node2].append((node1, cost))

## Step 1: Grid-based Search

Implement BFS and DFS on `regions.txt`. Use the grid to plan travel between hospitals (S) and outbreak zones (G). Keep track of explored nodes.

In [19]:
class Grid:
    def __init__(self, filename):
        with open(filename, 'r') as f:
            lines = f.readlines()
        self.rows, self.cols = map(int, lines[0].split())
        self.grid = [line.strip().split() for line in lines[1:]]
        self.start = None
        self.goal = None
        for i in range(self.rows):
            for j in range(self.cols):
                if self.grid[i][j] == 'S':
                    self.start = (i, j)
                elif self.grid[i][j] == 'G':
                    self.goal = (i, j)

    def bfs(self):
        from collections import deque
        queue = deque([self.start])
        visited = set([self.start])
        parent = {self.start: None}
        explored = 0
        while queue:
            current = queue.popleft()
            explored += 1
            if current == self.goal:
                path = []
                while current:
                    path.append(current)
                    current = parent[current]
                path.reverse()
                return path, explored
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = current[0] + di, current[1] + dj
                if 0 <= ni < self.rows and 0 <= nj < self.cols and self.grid[ni][nj] != 'X' and (ni,nj) not in visited:
                    visited.add((ni,nj))
                    queue.append((ni,nj))
                    parent[(ni,nj)] = current
        return None, explored

    def dfs(self):
        stack = [self.start]
        visited = set([self.start])
        parent = {self.start: None}
        explored = 0
        while stack:
            current = stack.pop()
            explored += 1
            if current == self.goal:
                path = []
                while current:
                    path.append(current)
                    current = parent[current]
                path.reverse()
                return path, explored
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = current[0] + di, current[1] + dj
                if 0 <= ni < self.rows and 0 <= nj < self.cols and self.grid[ni][nj] != 'X' and (ni,nj) not in visited:
                    visited.add((ni,nj))
                    stack.append((ni,nj))
                    parent[(ni,nj)] = current
        return None, explored

## Step 2: Heuristic Graph Search

Implement GBFS and A* search on `graph.txt` using `heuristics.txt`. Use priority queues and track path costs.

In [20]:
def read_graph(filename):
    graph = {}
    with open(filename, 'r') as f:
        lines = f.readlines()
    num_nodes = int(lines[0].strip())
    for line in lines[1:]:
        parts = line.strip().split()
        if len(parts) == 3:
            n1, n2, cost = parts[0], parts[1], int(parts[2])
            if n1 not in graph:
                graph[n1] = []
            if n2 not in graph:
                graph[n2] = []
            graph[n1].append((n2, cost))
            graph[n2].append((n1, cost))  # undirected
    return graph

def read_heuristic(filename):
    heuristic = {}
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                node, h = parts[0], int(parts[1])
                heuristic[node] = h
    return heuristic

def greedy_best_first_search(graph, heuristic, start, goal):
    import heapq
    pq = [(heuristic.get(start, 0), start, 0)]  # (h, node, g)
    visited = set()
    parent = {start: None}
    while pq:
        _, current, g = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)
        if current == goal:
            path = []
            while current:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path, g
        for neighbor, edge_cost in graph.get(current, []):
            if neighbor not in visited:
                heapq.heappush(pq, (heuristic.get(neighbor, 0), neighbor, g + edge_cost))
                parent[neighbor] = current
    return None, float('inf')

def a_star(graph, heuristic, start, goal):
    import heapq
    pq = [(heuristic.get(start, 0), start, 0)]  # (f, node, g)
    visited = set()
    parent = {start: None}
    g_cost = {start: 0}
    while pq:
        _, current, g = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)
        if current == goal:
            path = []
            while current:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path, g
        for neighbor, edge_cost in graph.get(current, []):
            new_g = g + edge_cost
            if neighbor not in g_cost or new_g < g_cost[neighbor]:
                g_cost[neighbor] = new_g
                f = new_g + heuristic.get(neighbor, 0)
                heapq.heappush(pq, (f, neighbor, new_g))
                parent[neighbor] = current
    return None, float('inf')

## Step 3: Minimax and Alpha-Beta

Use `game_state.json` to model MAX (control team) vs MIN (disease progression). Implement both standard Minimax and Alpha-Beta pruning.

In [21]:
class GameState:
    def __init__(self, resources, patients, turn):
        self.resources = resources.copy()
        self.patients = patients.copy()
        self.turn = turn

    def is_terminal(self):
        return self.turn > 5 or sum(self.patients.values()) <= 0

    def evaluate(self):
        # Assume initial patients: critical 50, moderate 200, mild 100
        treated = (50 - max(0, self.patients['critical'])) + (200 - max(0, self.patients['moderate'])) + (100 - max(0, self.patients['mild']))
        return treated if self.turn % 2 == 1 else -treated

    def get_actions(self):
        actions = []
        max_d = min(self.resources['doctors'], self.patients['critical'])
        max_n = min(self.resources['nurses'], self.patients['moderate'])
        max_v = min(self.resources['vaccines'], self.patients['mild'])
        for d in range(0, max_d + 1, 5):  # step 5 to reduce
            for n in range(0, max_n + 1, 10):
                for v in range(0, max_v + 1, 20):
                    actions.append({'doctors': d, 'nurses': n, 'vaccines': v})
        return actions

    def apply_action(self, action):
        new_resources = self.resources.copy()
        new_patients = self.patients.copy()
        new_resources['doctors'] -= action['doctors']
        new_resources['nurses'] -= action['nurses']
        new_resources['vaccines'] -= action['vaccines']
        new_patients['critical'] = max(0, new_patients['critical'] - action['doctors'])
        new_patients['moderate'] = max(0, new_patients['moderate'] - action['nurses'])
        new_patients['mild'] = max(0, new_patients['mild'] - action['vaccines'])
        # For MIN (disease), increase patients
        if self.turn % 2 == 0:
            new_patients['critical'] += 2
            new_patients['moderate'] += 5
            new_patients['mild'] += 10
        return GameState(new_resources, new_patients, self.turn + 1)

def minimax(state, depth, maximizingPlayer):
    if depth == 0 or state.is_terminal():
        return None, state.evaluate()
    if maximizingPlayer:
        max_eval = -float('inf')
        best_action = None
        for action in state.get_actions():
            _, eval_score = minimax(state.apply_action(action), depth - 1, False)
            if eval_score > max_eval:
                max_eval = eval_score
                best_action = action
        return best_action, max_eval
    else:
        min_eval = float('inf')
        best_action = None
        for action in state.get_actions():
            _, eval_score = minimax(state.apply_action(action), depth - 1, True)
            if eval_score < min_eval:
                min_eval = eval_score
                best_action = action
        return best_action, min_eval

def alpha_beta(state, depth, alpha, beta, maximizingPlayer):
    if depth == 0 or state.is_terminal():
        return None, state.evaluate()
    if maximizingPlayer:
        max_eval = -float('inf')
        best_action = None
        for action in state.get_actions():
            _, eval_score = alpha_beta(state.apply_action(action), depth - 1, alpha, beta, False)
            if eval_score > max_eval:
                max_eval = eval_score
                best_action = action
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return best_action, max_eval
    else:
        min_eval = float('inf')
        best_action = None
        for action in state.get_actions():
            _, eval_score = alpha_beta(state.apply_action(action), depth - 1, alpha, beta, True)
            if eval_score < min_eval:
                min_eval = eval_score
                best_action = action
            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return best_action, min_eval

## Step 4: Propositional Logic & Knowledge Base

Use `kb.txt` and `sensors.txt` / `rules.txt`. Implement CNF conversion, resolution, and dynamic rule evaluation.

In [22]:
class KnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.rules = []

    def add_fact(self, fact):
        self.facts.add(fact)

    def add_rule(self, rule):
        self.rules.append(rule)

    def resolve_query(self, query):
        # Simple forward chaining
        inferred = set(self.facts)
        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                if 'if' in rule and 'then' in rule:
                    premise, conclusion = rule.split('then')
                    premise = premise.replace('if', '').strip()
                    conditions = [c.strip() for c in premise.split('and')]
                    if all(c in inferred for c in conditions):
                        conc = conclusion.strip()
                        if conc not in inferred:
                            inferred.add(conc)
                            changed = True
        return query in inferred

## Step 5: First-Order Logic

Use `patients.txt` and `hospitals.txt` to model entities and relationships. Implement FOL evaluation and queries (e.g., are all critical patients assigned?).

In [23]:
class FOL_KB:
    def __init__(self):
        self.entities = {}  # id: attrs
        self.relationships = {}  # rel_type: list of (e1, e2)

    def add_entity(self, entity_type, entity_id, **attrs):
        self.entities[entity_id] = attrs

    def add_relationship(self, rel_type, entity1, entity2):
        if rel_type not in self.relationships:
            self.relationships[rel_type] = []
        self.relationships[rel_type].append((entity1, entity2))

    def query(self, statement):
        if statement == "all critical patients assigned":
            critical = [eid for eid, attrs in self.entities.items() if attrs.get('status') == 'critical']
            assigned = self.relationships.get('assigned', [])
            assigned_patients = set(e1 for e1, e2 in assigned)
            return all(p in assigned_patients for p in critical)
        return False

## Step 6: Probabilistic Reasoning

Use `probabilities.csv` to implement joint, marginal, conditional probabilities, Bayes Rule, and independence checks.

In [24]:
class ProbabilityEngine:
    def __init__(self):
        self.joint = {}  # (status, treatment): prob

    def load_csv(self, filename):
        import csv
        with open(filename, 'r') as f:
            reader = csv.reader(f)
            next(reader)  # header
            for row in reader:
                status, treatment, prob = row[0], row[1], float(row[2])
                self.joint[(status, treatment)] = prob

    def marginal(self, variables):
        if 'PatientStatus' in variables:
            statuses = {}
            for (s, t), p in self.joint.items():
                statuses[s] = statuses.get(s, 0) + p
            return statuses
        elif 'Treatment' in variables:
            treatments = {}
            for (s, t), p in self.joint.items():
                treatments[t] = treatments.get(t, 0) + p
            return treatments

    def conditional(self, given, target):
        if given == {'Treatment': 'ICU'} and target == 'PatientStatus':
            p_icu = sum(p for (s, t), p in self.joint.items() if t == 'ICU')
            cond = {}
            for (s, t), p in self.joint.items():
                if t == 'ICU':
                    cond[s] = p / p_icu if p_icu > 0 else 0
            return cond
        return {}

    def bayes(self, A, B):
        # P(A|B) where A is status, B is treatment
        p_a = 1/3  # assume uniform
        p_b_given_a = self.joint.get((A, B), 0)
        p_b = sum(p for (s, t), p in self.joint.items() if t == B)
        if p_b == 0:
            return 0
        return p_b_given_a * p_a / p_b

## Step 7: Main Integration

Combine all previous components into a single main() function that:

- Reads all files
- Runs grid search
- Runs GBFS/A*
- Runs Minimax / Alpha-Beta
- Evaluates logic rules and FOL queries
- Computes probabilistic risk
- Writes all results to `output.txt`

In [25]:
def write_output(results, filename):
    with open(filename, 'w') as f:
        for key, value in results.items():
            f.write(f"{key}: {value}\n")

def main():
    results = {}

    # Step 1: Grid search
    grid = Grid("regions.txt")
    bfs_path, bfs_explored = grid.bfs()
    dfs_path, dfs_explored = grid.dfs()
    results['BFS Path'] = bfs_path
    results['BFS Explored'] = bfs_explored
    results['DFS Path'] = dfs_path
    results['DFS Explored'] = dfs_explored

    # Step 2: Graph search
    graph = read_graph("graph.txt")
    heuristic = read_heuristic("heuristics.txt")
    gbfs_path, gbfs_cost = greedy_best_first_search(graph, heuristic, 'A', 'T')
    astar_path, astar_cost = a_star(graph, heuristic, 'A', 'T')
    results['GBFS Path'] = gbfs_path
    results['GBFS Cost'] = gbfs_cost
    results['A* Path'] = astar_path
    results['A* Cost'] = astar_cost

    # Step 3: Minimax / Alpha-Beta
    import json
    with open("game_state.json", 'r') as f:
        data = json.load(f)
    state = GameState(data['resources'], data['patients'], data['turn'])
    minimax_action, minimax_value = minimax(state, 3, True)
    alphabeta_action, alphabeta_value = alpha_beta(state, 3, -float('inf'), float('inf'), True)
    results['Minimax Action'] = minimax_action
    results['Minimax Value'] = minimax_value
    results['AlphaBeta Action'] = alphabeta_action
    results['AlphaBeta Value'] = alphabeta_value

    # Step 4: Knowledge Base & Rules
    kb = KnowledgeBase()
    with open('kb.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if 'if' in line:
                kb.add_rule(line)
            else:
                kb.add_fact(line)
    query_result = kb.resolve_query("patient_1 assigned_to hospital_1")
    results['KB Query'] = query_result

    # Step 5: First-Order Logic
    fol_kb = FOL_KB()
    with open('patients.txt', 'r') as f:
        for line in f:
            parts = line.strip().split('(')[1].split(')')[0].split(',')
            pid, status = parts[0].strip(), parts[1].strip()
            fol_kb.add_entity('Patient', pid, status=status)
    with open('hospitals.txt', 'r') as f:
        for line in f:
            parts = line.strip().split('(')[1].split(')')[0].split(',')
            hid, cap = parts[0].strip(), parts[1].strip().split('=')[1]
            fol_kb.add_entity('Hospital', hid, capacity=int(cap))
    # Assume some assignments
    fol_kb.add_relationship('assigned', 'p1', 'h1')
    fol_kb.add_relationship('assigned', 'p4', 'h2')
    fol_query = fol_kb.query("all critical patients assigned")
    results['FOL Query'] = fol_query

    # Step 6: Probabilistic Reasoning
    prob_engine = ProbabilityEngine()
    prob_engine.load_csv("probabilities.csv")
    marginal_status = prob_engine.marginal(['PatientStatus'])
    conditional = prob_engine.conditional({'Treatment': 'ICU'}, 'PatientStatus')
    bayes_prob = prob_engine.bayes('critical', 'ICU')
    results['Marginal Status'] = marginal_status
    results['Conditional'] = conditional
    results['Bayes'] = bayes_prob

    # Step 7: Write integrated output
    write_output(results, "output.txt")

if __name__ == "__main__":
    main()

## Step 8: Questions

1. Compare BFS, DFS, GBFS, and A* for speed, memory, and path quality in this scenario.
2. How does Alpha-Beta pruning improve resource allocation decisions?
3. Discuss the advantages of modeling with propositional vs first-order logic.
4. How does probabilistic reasoning influence decision-making under uncertainty?
5. Suggest one real-world improvement to the integrated system.

In [26]:
# Answers
#
# 1. Compare BFS, DFS, GBFS, and A* for speed, memory, and path quality in this scenario.
# BFS: Guarantees shortest path in unweighted grid, explores level by level, high memory for large grids.
# DFS: May find a path quickly but not optimal, low memory, can get stuck in deep paths.
# GBFS: Fast, uses heuristic, but may not find optimal path if heuristic is not admissible.
# A*: Optimal and complete, combines cost and heuristic, more memory than GBFS but better quality.
#
# 2. How does Alpha-Beta pruning improve resource allocation decisions?
# It reduces the number of nodes evaluated in the minimax tree by pruning branches that won't affect the final decision, making it more efficient for deeper searches, leading to better resource allocation under time constraints.
#
# 3. Discuss the advantages of modeling with propositional vs first-order logic.
# Propositional logic is simpler, deals with true/false statements, good for rule-based systems. First-order logic handles objects, relations, and quantification, more expressive for complex domains like patient assignments.
#
# 4. How does probabilistic reasoning influence decision-making under uncertainty?
# It allows calculating probabilities of outcomes, helps in risk assessment, and supports decisions based on expected values rather than certainties, crucial for treatments with varying success rates.
#
# 5. Suggest one real-world improvement to the integrated system.
# Integrate machine learning to predict outbreak patterns based on sensor data, allowing proactive resource allocation.